# Assignment — L01: Sarah's Secondment to Lakeside Bank

> *Sarah Chen · Customer Experience Analyst, NorthStar Retail, on secondment to Lakeside Bank · February 2023.*

This is your **self-study practice** — you'll re-apply what you learned in class to a completely different domain (banking, not retail). The notebook has four parts:

1. **Meet Tom** — the new context and the new problem.
2. **What's different here?** — pause and notice the differences before writing code.
3. **Concept refresher** — brief recap of the three things from this lesson you'll lean on.
4. **Practice** — three tiers of guided practice + three independent exercises + sample solutions.

Work through it in order. Sample solutions live at the bottom — only look after you've tried.

---

## Part 1 — Meet Tom

It's Monday, February 6th, 2023. Sarah has just been told by Priya:

> *"Priya: We have a co-brand partnership with Lakeside Bank starting this quarter. Tom Bradley, their Head of Analytics, needs a hand. You did great work on our reviews — go help him for two days. Consider it professional development."*

Sarah arrives at Lakeside's Seattle office. Tom's office is an unlabelled door with a "Please knock" sign. He's sitting in front of two monitors, both open to spreadsheets.

> *Tom: "Thanks for coming over. Honestly I have no idea what Priya told you — I probably don't need an ML person for this. But I have a problem. Customers are writing complaints in our mobile-banking app's feedback form. I've got about 8,000 of them from the last quarter. I've been meaning to categorise them. Every time I've tried, I've given up halfway through because I just can't read 8,000 things."*

> *Tom: "I don't know what 'machine learning' actually means. I just need to know: can you help me figure out which complaints are about fees, which are about the app being broken, and which are about something else?"*

Sarah smiles. This sounds familiar.

---

## Part 2 — What's different here?

Before writing any code, Sarah takes out her notebook and thinks.

**Stop and reflect (2 minutes).** Compare Tom's problem to her NorthStar problem from last week. They look similar at first glance — text classification. But there are differences.

Write down in your head (or in a note):

1. What **category of ML** does Tom's problem fit into (supervised, unsupervised, reinforcement)?
2. What's different about Tom's data versus NorthStar's reviews?
3. What would **success** look like for Tom? What would failure look like?
4. Is there an off-the-shelf ML model Sarah can use, like the DistilBERT pipeline she used at NorthStar?

*(Sample answers at the bottom of this notebook — look after you try.)*


(26 Jun 2026 PM) My attempt:
1. Would say reinforcement since it will only be 3.3 we start doing supervised ML.
2. Likely keywords may differ in the context on banking versus retail. Retail also have external elements like delivery, where banking does not.
3. Identify what aspects resulted in customer's praises and also the negative feedback for improvement, and remediation.
4. I believe there is on non-retail banking one?

---

## Part 3 — Concept refresher

Pull forward three things from this lesson:

**Framing first.** Before touching code, Sarah should pin down: *What decision will Tom make with the output? Which is more expensive — a false "fees" or a false "app broken"?* If no one acts on the output, the model is zero value.

**The three categories.** Tom has text without labels. If he needs categories he defined in advance (fees / app / other), he can label a small sample and train a **supervised** classifier. If he's open to letting the data suggest groupings, he can use **unsupervised** clustering. Both are valid.

**The workflow.** Even a "simple" model needs the 7 steps: frame, collect, clean, train, evaluate, deploy, monitor. Skipping evaluation is how Priya's doubt becomes expensive.

---

## Part 4 — Practice

Three **tiers** of practice (graduated scaffolding) followed by three **exercises** (independent work). Sample solutions at the end.

Don't skip Tier 1 even if it looks easy — the pattern you practice there is the pattern you'll reuse in Tier 3.

---

## 📚 Choose your track

This assignment has **two tracks**. Pick **one** based on your background — you don't need to do both.

| Track | Who it's for | What you'll do |
|---|---|---|
| **🟢 Foundational Track** | Learners new to ML / programming | Tier 1 (run sentiment) + Tier 2 (pick ML category) |
| **🔵 Advanced Track** | Learners with prior ML background | Tier 3 framing memo + the 3 independent exercises |

If you're unsure, start with the **Foundational Track**. If it feels easy, skip ahead to the **Advanced Track** — both tracks cover the same lesson outcomes; only the scaffolding differs.

---


---

# 🟢 Foundational Track

> *No prior ML background needed. The cells below are scaffolded — read the worked example, then fill in the blanks. Hints are included.*

---


In [1]:
import os
# Must be set before importing transformers to prevent tokenizer deadlock on macOS
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from transformers import pipeline
import warnings
import random

# macOS guard: cap PyTorch threads to 1 to keep the OpenMP scheduler from
# hanging the sentiment loop. Harmless on Linux/Windows.
import torch
torch.set_num_threads(1)

warnings.filterwarnings("ignore")
random.seed(42)

# Load the same DistilBERT pipeline we used in 02_what_is_ml.ipynb and
# 04_ml_workflow.ipynb. Cached after the first run.
print("Loading pre-trained sentiment model... (cached after first run)")
classifier = pipeline("sentiment-analysis")
print("Ready.")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading pre-trained sentiment model... (cached after first run)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Ready.


### Tier 1 — Guided: Sentiment on Tom's feedback (read, run, predict)

Here are 6 real-looking Lakeside mobile-banking feedback entries. We'll run the DistilBERT sentiment pipeline on them — just like Sarah did at NorthStar.

DistilBERT returns a **label** (`POSITIVE` or `NEGATIVE`) and a **confidence score** between 0 and 1. The label answers *"which side is this on?"*; the confidence answers *"how sure is the model?"* Low-confidence predictions are exactly the cases worth flagging for human review.

In [2]:
lakeside_feedback = [
    "App is genuinely great. Transferring money is easier than my old bank.",
    "I've been charged a $32 overdraft fee when I wasn't overdrawn. Called support, was on hold 45 minutes.",
    "The fingerprint login never works on my phone. Have to type the password every time.",
    "Thanks for the interest rate bump on savings accounts this month. Surprised and happy.",
    "Tried to deposit a check by photo — it crashed. Three times. How is this 2023.",
    "Fees are reasonable. Customer service has been responsive. Overall fine.",
]

# The pipeline returns a dict like {"label": "POSITIVE", "score": 0.998}.
for i, fb in enumerate(lakeside_feedback, 1):
    result = classifier(fb)[0]
    label = result["label"].lower()
    confidence = result["score"]
    review_flag = "  ← review me" if confidence < 0.85 else ""
    print(f"{i}. [{label:8} {confidence:.2%}]  {fb[:60]}...{review_flag}")

1. [positive 99.96%]  App is genuinely great. Transferring money is easier than my...
2. [negative 99.61%]  I've been charged a $32 overdraft fee when I wasn't overdraw...
3. [negative 99.95%]  The fingerprint login never works on my phone. Have to type ...
4. [positive 99.99%]  Thanks for the interest rate bump on savings accounts this m...
5. [negative 99.94%]  Tried to deposit a check by photo — it crashed. Three times....
6. [positive 99.05%]  Fees are reasonable. Customer service has been responsive. O...


**Reflection (Tier 1) — answer in your head or in a markdown cell:**

1. Which feedback entries did the model get clearly right? Clearly wrong?
2. Would Tom be happy with these outputs, or does he need more than sentiment?
3. What's the next thing you'd build for Tom after this?


(26 Jun 2026 PM) My Attempt:
1. All 6 samples returned the desired output.
2. Since all 6 yielded 100% accuracy, Tom could get more assurance by running another round of samples more than 6.
3. If supervised learning has such high accuracy already, maybe can build a unsupervised to cluster, on categories such as security, app, customer service to get a more catered resolution.

### Tier 2 — Partial: Pick the right ML category for Tom

Tom wants complaints grouped into **fees / app / other**. We need to decide: supervised or unsupervised? Fill in the blanks.

<details><summary>💡 Hint</summary>

Does Tom have labelled examples (already tagged as fees / app / other) already? Or does he need to label some himself first?

</details>

In [3]:
# Fill in the blanks by replacing each ___ with your answer.
# Use the strings: 'supervised', 'unsupervised', 'reinforcement', or a category like 'fees'.

# 1. Tom wants outputs labelled fees / app / other.
# If he has no labelled examples yet, which category of ML does he pick *to get started*?
choice_getting_started = 'unsupervised'  # your answer: 'supervised', 'unsupervised', or 'reinforcement'

# 2. Once Tom has labelled (say) 300 complaints by hand into fees / app / other,
# he can train a different category of model to handle the remaining 7,700.
choice_after_labelling = 'supervised'  # your answer

# Based on Gemini AI to work this out, unsupervised to cluster them ito the "right" labels. Then with the results, to supervised the remaining reviews into the right categories.

# 3. Which category is Sarah NOT going to use on Tom's problem?
choice_not_used = 'reinforcement'  # your answer

# seems like reinforcement is "on-the-go" type, like Chess.

# Check yourself:
assert choice_getting_started in {'supervised', 'unsupervised', 'reinforcement'}
assert choice_after_labelling in {'supervised', 'unsupervised', 'reinforcement'}
assert choice_not_used in {'supervised', 'unsupervised', 'reinforcement'}
print('Answers recorded:')
print(f'  1. To get started with no labels: {choice_getting_started}')
print(f'  2. After Tom labels 300 by hand: {choice_after_labelling}')
print(f'  3. Not used on this problem:      {choice_not_used}')

Answers recorded:
  1. To get started with no labels: unsupervised
  2. After Tom labels 300 by hand: supervised
  3. Not used on this problem:      reinforcement


**Sample answers (check after trying):**
1. **unsupervised** — with no labels, Sarah would cluster the feedback and show Tom the groupings.
2. **supervised** — once there are labels, train a classifier.
3. **reinforcement** — no reward signal here; not applicable.

**Tier 2 reflection:** Why does it matter that Sarah is honest with Tom about which category she's using, rather than just saying "I'm using ML"?


(26 Jun 2026 PM): 
Tom would not understand the technical jargons, so it is better for Sarah to be transparent for alignment.

---

# 🔵 Advanced Track

> *For learners with prior ML background. Minimal scaffolding — you decide the approach. You're welcome to peek at the Foundational Track above for reference.*

---


### Tier 3 — Open: Draft a framing memo for Tom

No code for this tier — this is pure framing practice, which is the most valuable skill you'll build in this course.

In a markdown cell below, write a **half-page "framing memo"** you would send to Tom before you build anything. The memo must cover:

- **The question** — what is Tom actually trying to answer?
- **The decision** — who acts on the output, and what do they do with it?
- **Success criteria** — how will Tom know the model is good enough to use?
- **Cost of wrong answers** — what happens if the model miscategorises 10% of complaints? 30%?
- **Data you need** — what does Tom need to send you to get started?

Write 150–300 words. There is no single right answer. A sample memo is at the bottom of this notebook.

> _Your framing memo here_

> _(Replace this line with your 150–300 word memo.)_
My Attempt (26 Jun 2026):

1. Tom wants to find out what are the improvements that can be done based on the customers' feedback/reviews.
2. The initial labelling from samples could be done by Sarah and gets cleared with Tom (and/or other team heads) - cluster samples and from the results do the labelling.
3. With a high confidence level and a range on the accuracy.
4. The cost for wrong answers will result in additional monies and efforts spent unnecessarily.
5. Enough data for training.

---

## 5. Assignment Exercises (independent)

Three exercises. Each has a scenario and a numbered task list. A blank code cell and a blank interpretation cell follow each. Sample solutions at the very bottom — do not look until you've attempted each exercise.

These exercises are still set at Lakeside Bank. Tom has sent Sarah more data.

### Exercise 1 — Classify a new batch

**Scenario:** Tom has sent 10 new feedback entries. Sarah needs to classify sentiment for the Monday team meeting.

**Tasks:**
1. Run the DistilBERT sentiment pipeline (`classifier`, loaded in the imports cell) on each of the 10 entries in `ex1_feedback` below.
2. Print a summary count: how many `positive`, how many `negative`? Then count how many predictions came in **below 85% confidence** — those are the entries Sarah should review by hand.
3. Identify the entry classified as **negative with the highest confidence** (the "most clearly negative") and print it.
4. Write a 2-sentence interpretation for Tom.

In [4]:
ex1_feedback = [
    'The new UI looks great but the menu is confusing now.',
    'Incredible customer support. Got my issue resolved in 6 minutes.',
    'They charged me twice for the same wire transfer. Still waiting for a refund after 3 weeks.',
    'Fine.',
    'Absolutely furious. This app has deleted my transaction history twice.',
    'Thanks to whoever fixed the deposit bug. It works again.',
    "I've been a customer for 12 years and the fees keep going up. Time to switch.",
    'Nothing special, nothing terrible.',
    'Best banking app I have used, including the ones at my old bank.',
    'Password reset flow is broken on iOS.',
]

# Your code here

# Add counter
from collections import Counter

counts = Counter()

# To use Tier 1 code above
for i, fb in enumerate(ex1_feedback, 1):
    result = classifier(fb)[0]
    label = result["label"].lower()
    confidence = result["score"]
    review_flag = "  ← review me" if confidence < 0.85 else ""
    print(f"{i}. [{label:8} {confidence:.2%}]  {fb[:60]}...{review_flag}")

    counts[label] += 1

# print summary count on positive and negative
print("\nSummary of sentiment analysis:")
print("\n" + "="*30)
print("SUMMARY OF FEEDBACK")
print("="*30)
print(f"Positive: {counts['positive']}")
print(f"Negative: {counts['negative']}")
print(f"Total Processed: {sum(counts.values())}")

1. [negative 99.79%]  The new UI looks great but the menu is confusing now....
2. [positive 99.95%]  Incredible customer support. Got my issue resolved in 6 minu...
3. [negative 99.84%]  They charged me twice for the same wire transfer. Still wait...
4. [positive 99.98%]  Fine....
5. [negative 99.95%]  Absolutely furious. This app has deleted my transaction hist...
6. [positive 99.77%]  Thanks to whoever fixed the deposit bug. It works again....
7. [positive 92.76%]  I've been a customer for 12 years and the fees keep going up...
8. [negative 99.62%]  Nothing special, nothing terrible....
9. [positive 99.95%]  Best banking app I have used, including the ones at my old b...
10. [negative 99.90%]  Password reset flow is broken on iOS....

Summary of sentiment analysis:

SUMMARY OF FEEDBACK
Positive: 5
Negative: 5
Total Processed: 10


**Your interpretation for Tom (2 sentences):**

_Write here._
(26Jun2026 PM)
Based on sampling, the result is a 50% postive-negative. However, there are feedback that are subjective, and may skew the desired outcome. We may need to place them apart. We may enlist assistance from customer service to determine if the labelling is correct.

### Exercise 2 — Decide the category

**Scenario:** Tom has two new problems on his desk.

- **Problem A:** Tom wants to predict which mortgage applications will default. He has 10 years of history with labelled outcomes (defaulted or not).
- **Problem B:** Tom wants to understand whether Lakeside's 180,000 customers fall into natural "types" for a new product launch. He has no pre-existing types in mind.

**Tasks:**
1. For each problem, state which category of ML applies (supervised / unsupervised / reinforcement) and why.
2. For each problem, give one example of a **feature** and one example of a **label** (or "no label" if unsupervised).
3. For Problem A, describe one risk of getting this wrong.

**Your answer:**

_Write here._

(26Jun2026 PM)
1.
Problem A: Supervised, since there is historical data for training, and outcome is binary.
Problem B: Unsupervised since there is no existing labellings/categorisations.

2. 
Problem A: Income, Default_Flag
Problem B: Mortgage_Type, "no label"

3. Risk for Problem A is not identifying default correctly (false negative). If the application is approved and the applicant default, it adds pressure to the bank, and will be scrutinised by the central bank

### Exercise 3 — Draft an ML workflow for Tom's complaint categoriser

**Scenario:** Tom has decided to go ahead with Sarah's proposal: build a classifier that sorts his 8,000 complaints into fees / app / other. He has agreed to hand-label 400 complaints over a week. He wants to present the plan to his boss on Friday.

**Tasks:**
1. Write down the 7 steps of the ML workflow specifically for this project (each one specific to Tom's data, not generic).
2. Identify the single step where you are most likely to spend the most time, and explain why.
3. Describe one thing that could go wrong in production that would only show up in Step 7 (monitoring).

**Your answer:**

_Write here._

(26Jun2026 PM)
1. 7 Steps
Frame -> Collect -> Clean -> Train -> Evaluate -> Deploy -> Monitor
# See Notebook 04

i. Frame: Tom to determine the business value of the reviews and the subsequent actions.
ii. Collect: Ready 8000 reviews, and more going forward
iii. Clean: Missing values to fill, duplicating reviews, correct labelling
iv. Train: To use distilbert as initial, and another ML model to train for deployment?
v. Evaluate: To verify the accuracy of prediction/categorisation
vi. Deploy: Once ready, to push for utility.
vii. Monitor: Ensure that model is still relevant as time passes

2. Cleaning is the time consuming step as preparation usualy takes the longest, to ensure cleanliness in for a clean output, which will then be repeated during deployment.

3. New features introduced or old features obsolete?


---

## 6. Submission Checklist

Before you submit, verify:

- [ ] Every code cell runs without error
- [ ] Every exercise has both your code and your written interpretation
- [ ] The framing memo in Tier 3 is 150–300 words
- [ ] You attempted each exercise before looking at the sample solution
- [ ] You can explain, in 1 sentence, the single biggest thing you learned from doing this

---

## 7. Sample Solutions

**Do not read until you've attempted the exercises.**

### Reflection — Tom's problem vs NorthStar's (sample angles)

1. **Category:** Unsupervised if Tom has no labels; supervised if he's willing to label a sample first. Both are valid.
2. **What's different:** Tom's feedback is written in a mobile app (so shorter, more typos, often angrier) vs NorthStar's product reviews. Tom doesn't have pre-existing categories, whereas NorthStar's sentiment categories (positive/negative) are well-known.
3. **Success:** Tom can confidently say "60% of complaints this quarter were about fees" — not by reading all 8,000, but by trusting the model's groupings.
4. **Failure:** the groupings don't match what product managers would intuitively call groups, or the model misses a surge in a new category (e.g., security concerns).
5. **Off-the-shelf:** The DistilBERT pipeline can score *sentiment* (positive/negative), but it cannot do *topic* clustering (fees vs app vs other). Tom needs something different — hinted at in L05 (clustering) and L09 (NLP topic models).

### Sample — Tier 3 framing memo

> **To:** Tom Bradley, Head of Analytics, Lakeside Bank
> **From:** Sarah Chen, NorthStar Retail (on secondment)
> **Date:** February 8, 2023
> **Subject:** Complaint Categorisation — Framing Memo

> **The question.** Can we categorise Lakeside's mobile-banking complaints into buckets that make sense to the product and operations teams — so trends are visible at a glance rather than lost in a spreadsheet?

> **The decision.** Product Ops will use the weekly breakdown to prioritise bug fixes and fee policy changes. Customer Service will track category volumes as a leading indicator of process issues.

> **Success criteria.** (a) ≥80% of complaints are auto-categorised into fees / app / other with human-reviewer agreement. (b) A fourth category ('other — investigate') catches anomalies rather than hiding them. (c) The categoriser processes a new complaint in under 2 seconds (no waiting queue).

> **Cost of wrong answers.** A fees complaint miscategorised as 'app' looks like a bug — dev team wastes time. An app complaint miscategorised as 'fees' looks like a policy issue — account team escalates for nothing. Worst case: a security complaint lost in 'other' — this is a material risk and we must review 'other' weekly by hand.

> **Data request.** 8,000 complaints, 400 labelled by an experienced CS agent over one week. Plus category definitions in writing from Ops.

> — Sarah

### Sample solution — Exercise 1

In [5]:
# Exercise 1 sample solution
results = [(fb, classifier(fb)[0]) for fb in ex1_feedback]

positive = sum(1 for _, r in results if r["label"] == "POSITIVE")
negative = sum(1 for _, r in results if r["label"] == "NEGATIVE")
low_confidence = sum(1 for _, r in results if r["score"] < 0.85)

print(f"Positive: {positive}")
print(f"Negative: {negative}")
print(f"Low-confidence (<85%) — flag for human review: {low_confidence}")

# "Most clearly negative" = the NEGATIVE prediction with the highest confidence.
negatives = [(fb, r["score"]) for fb, r in results if r["label"] == "NEGATIVE"]
if negatives:
    most_negative_fb, most_negative_score = max(negatives, key=lambda pair: pair[1])
    print()
    print(f"Most clearly negative ({most_negative_score:.2%} confidence): {most_negative_fb}")

Positive: 5
Negative: 5
Low-confidence (<85%) — flag for human review: 0

Most clearly negative (99.95% confidence): Absolutely furious. This app has deleted my transaction history twice.


**Sample interpretation for Tom:**

> The Monday batch skews negative: most entries express frustration, concentrated around billing (double-charged wires, rising fees) and specific app bugs (iOS password reset, transaction history loss). The single most urgent entry is the **deleted-transaction-history** complaint — flagged confidently as negative — suggest Ops triage it immediately. Any low-confidence entries (e.g. "Fine." or "Nothing special, nothing terrible.") should be skimmed by a human rather than acted on, since DistilBERT is forced to pick a side even when the text is genuinely ambiguous.

### Sample solution — Exercise 2

**Problem A (mortgage default prediction):**
- Category: **Supervised** learning (classification). Tom has historical applications with known outcomes.
- Example feature: applicant's credit score. Example label: defaulted (yes/no) within 24 months.
- Risk: a biased training set (e.g., under-represented neighbourhoods) could produce a model that systematically denies creditworthy applicants — a regulatory and reputational problem. This is why L04's failure chapter exists.

**Problem B (customer typing):**
- Category: **Unsupervised** learning (clustering). No pre-existing types.
- Example feature: 12-month transaction spend by category. No label — the goal is to discover structure.
- This is L05's terrain — and Sarah will encounter it in her own NorthStar work a year from now.

### Sample solution — Exercise 3

**The 7 steps, specific to Tom:**

1. **Frame.** Agree with Tom's Ops team on exact categories (fees / app / other / unknown) and what action each one triggers. Decide a minimum confidence threshold below which the model says 'not sure, review manually'.
2. **Collect.** Export the 8,000 complaints from the feedback-form database; get 400 of them labelled by a senior CS agent over one week.
3. **Clean.** Normalise punctuation, remove personal info (PII), drop empty or 1-word entries, spot-check for duplicates.
4. **Train.** Fit a text classifier on the 400 labelled entries — start with a simple baseline (e.g., TF-IDF + logistic regression) before trying anything heavier.
5. **Evaluate.** Hold out ~80 of the 400 labelled entries as a test set. Measure accuracy, precision, and recall for each category. Must meet Tom's success criteria before shipping.
6. **Deploy.** Wire the model into the weekly complaint dashboard. The dashboard shows counts, trends, and a list of 'low confidence — please review' items.
7. **Monitor.** Track weekly category distributions. If the 'other' bucket grows week over week, that's a drift signal — re-label a fresh sample and retrain.

**Most time-consuming step:** Step 3 (Clean) or Step 2 (getting labels). Labelling 400 entries well takes days — and clean data is worth more than a clever model on messy data.

**Step 7 risk:** Lakeside launches a new product feature mid-quarter that generates an entirely new category of complaint ('biometric login glitch'). The existing model has never seen this, so it stuffs everything into 'other'. Without monitoring, Tom's dashboard looks stable — but a real issue is hiding in the 'other' bucket.

---

## ✅ End of Assignment

You've just applied L01 to a fresh domain.

**In this assignment you:**
- Experienced the concept in a new domain (banking, not retail)
- Reflected on what was different
- Re-connected to theory
- Applied the workflow end to end

**Next lesson — L02: Priya's doubt from the Part 3 closing question.** *How sure are we?* That question is the reason probability and statistics exist.